# Chapter 11: Probabilistic Graphical Models

```{admonition} Learning Objectives
:class: tip
- Understand probabilistic graphical models
- Master Bayesian networks and conditional probability
- Apply inference algorithms (Variable Elimination, Belief Propagation)
- Implement Hidden Markov Models (Forward, Backward, Viterbi)
- Learn parameters with EM algorithm
- Model uncertainty in AI systems
```

```{epigraph}
Probabilistic graphical models provide a principled framework for reasoning under uncertainty by representing complex joint distributions compactly.

-- Daphne Koller & Nir Friedman
```

## 11.1 Introduction

**Probabilistic Graphical Models (PGMs)** represent complex probability distributions using graph structures.

### Why Probabilistic Models?

**Real-world uncertainty**:
- Noisy sensors
- Incomplete information
- Multiple possible explanations
- Ambiguous data

**Traditional AI limitations**:
- Logic: All-or-nothing reasoning
- Search: Assumes complete knowledge
- Need: Quantify uncertainty

### Graph Structure + Probability

**PGMs combine**:
1. **Graph theory**: Structure and independence
2. **Probability theory**: Uncertainty quantification
3. **Algorithms**: Efficient inference and learning

### Types of PGMs

**1. Directed Models** (Bayesian Networks):
- DAG structure
- Conditional probabilities
- Causal interpretation

**2. Undirected Models** (Markov Networks):
- Undirected graph
- Potential functions
- Symmetric relationships

**3. Temporal Models**:
- Hidden Markov Models
- Dynamic Bayesian Networks
- Sequential data

### Applications

- **Medical diagnosis**: Symptom-disease relationships
- **Computer vision**: Image segmentation, object recognition
- **Natural language**: Part-of-speech tagging, parsing
- **Robotics**: Localization, SLAM
- **Bioinformatics**: Gene regulatory networks
- **Speech recognition**: Acoustic modeling

## 11.2 Bayesian Networks

**Bayesian Network**: Directed acyclic graph (DAG) representing conditional dependencies among random variables.

### 11.2.1 Structure

**Components**:
- **Nodes**: Random variables $X_1, X_2, ..., X_n$
- **Edges**: Direct probabilistic dependencies
- **CPDs**: Conditional Probability Distributions at each node

**Terminology**:
- **Parents** $Pa(X_i)$: Direct predecessors of $X_i$
- **Children** $Ch(X_i)$: Direct successors of $X_i$
- **Ancestors** $Anc(X_i)$: All nodes with path to $X_i$
- **Descendants** $Desc(X_i)$: All nodes reachable from $X_i$

### 11.2.2 Joint Probability Factorization

**Chain rule for Bayesian networks**:

$$P(X_1, X_2, ..., X_n) = \prod_{i=1}^{n} P(X_i | Pa(X_i))$$

**Key insight**: Each variable depends only on its parents

**Example**: For network $A \to B \to C$ with $A \to C$:
$$P(A,B,C) = P(A) \cdot P(B|A) \cdot P(C|A,B)$$

### 11.2.3 Conditional Independence

**Local Markov Property**: Each variable is conditionally independent of its non-descendants given its parents

$$X_i \perp NonDesc(X_i) | Pa(X_i)$$

**D-separation**: General criterion for conditional independence

Three basic structures:

**1. Chain**: $X \to Y \to Z$
- $X \perp Z | Y$ (Y blocks path)

**2. Common Parent**: $X \leftarrow Y \to Z$
- $X \perp Z | Y$ (Y blocks path)

**3. V-structure** (collider): $X \to Y \leftarrow Z$
- $X \perp Z$ (unconditionally independent)
- $X \not\perp Z | Y$ (conditioning creates dependence!)

### 11.2.4 Example: Medical Diagnosis

```
Network:
  Smoking → Lung Cancer → X-ray
     ↓
  Bronchitis
```

**Variables**:
- $S$: Smoking (True/False)
- $L$: Lung Cancer (True/False)
- $B$: Bronchitis (True/False)
- $X$: Abnormal X-ray (True/False)

**Joint distribution**:
$$P(S,L,B,X) = P(S) \cdot P(L|S) \cdot P(B|S) \cdot P(X|L)$$

**CPD tables** (example values):

$P(S=T) = 0.3$

| $S$ | $P(L=T|S)$ |
|-----|-----------|
| T   | 0.1       |
| F   | 0.01      |

| $S$ | $P(B=T|S)$ |
|-----|-----------|
| T   | 0.6       |
| F   | 0.3       |

| $L$ | $P(X=T|L)$ |
|-----|-----------|
| T   | 0.9       |
| F   | 0.2       |

## 11.3 Inference in Bayesian Networks

**Inference**: Compute probability of query variables given evidence

### 11.3.1 Types of Inference

**1. Marginal Inference**:
$$P(X) = \sum_{Y_1, ..., Y_k} P(X, Y_1, ..., Y_k)$$

**2. Conditional Inference**:
$$P(X|E=e) = \frac{P(X, E=e)}{P(E=e)}$$

**3. MAP Inference** (Maximum A Posteriori):
$$\arg\max_x P(X=x|E=e)$$

### 11.3.2 Enumeration (Brute Force)

**Idea**: Sum over all possible assignments

```
Algorithm: ENUMERATION-ASK(query X, evidence e, bn)

begin
    Q ← distribution over X
    
    for each value x_i of X do
        // Extend evidence with X = x_i
        Q[x_i] ← ENUMERATE-ALL(bn.VARS, {e, X=x_i})
    
    // Normalize
    return NORMALIZE(Q)
end

Algorithm: ENUMERATE-ALL(vars, e)

begin
    if vars is empty then
        return 1.0
    
    Y ← FIRST(vars)
    
    if Y has value y in e then
        // Use evidence value
        return P(y | Pa(Y)) × ENUMERATE-ALL(REST(vars), e)
    else
        // Sum over all values
        return ∑_y P(y | Pa(Y)) × ENUMERATE-ALL(REST(vars), {e, Y=y})
end
```

**Complexity**: $O(n \cdot 2^n)$ for $n$ Boolean variables

**Problem**: Exponential in number of variables

### 11.3.3 Variable Elimination

**Key insight**: Exploit factorization to avoid redundant computation

**Factors**: Functions over subsets of variables
- $f(X_1, ..., X_k)$: Table of values

**Operations on factors**:

**1. Product**: $f_3 = f_1 \times f_2$
$$f_3(X,Y,Z) = f_1(X,Y) \cdot f_2(Y,Z)$$

**2. Marginalization** (Sum out variable):
$$f_{-Y}(X,Z) = \sum_y f(X, Y=y, Z)$$

```
Algorithm: VARIABLE-ELIMINATION(query X, evidence e, bn, ordering)

begin
    // Initialize factors from CPDs
    factors ← []
    for each variable V in bn do
        factors.append(MAKE-FACTOR(V, Pa(V), e))
    
    // Process variables in order
    for each hidden variable H in ordering do
        // Find factors mentioning H
        relevant ← [f for f in factors if H in f.vars]
        
        // Remove from factor list
        factors ← factors - relevant
        
        // Multiply relevant factors
        product ← MULTIPLY(relevant)
        
        // Sum out H
        new_factor ← SUM-OUT(H, product)
        
        // Add back to list
        factors.append(new_factor)
    
    // Final result
    result ← MULTIPLY(factors)
    return NORMALIZE(result)
end
```

**Complexity**: Depends on elimination ordering
- Best case: $O(n \cdot 2^w)$ where $w$ is tree-width
- Worst case: Still exponential

**Choosing elimination order**:
- Heuristic: Eliminate variable that creates smallest factor
- Min-fill: Minimize edges added to graph
- Finding optimal order is NP-hard

### 11.3.4 Example: Variable Elimination

**Query**: $P(B|J=T, M=T)$ in Burglary network

```
Network: B → A ← E
         ↓   ↓
         J   M
```

**Steps**:
1. Set evidence: $J=T, M=T$
2. Eliminate $E$: $f_1(A) = \sum_e P(E=e) \cdot P(A|B,E=e)$
3. Eliminate $A$: $f_2(B) = \sum_a f_1(A=a) \cdot P(J=T|A=a) \cdot P(M=T|A=a)$
4. Multiply: $f_3(B) = P(B) \cdot f_2(B)$
5. Normalize: $P(B|J=T,M=T) = \alpha \cdot f_3(B)$

## 11.4 Hidden Markov Models (HMMs)

**HMM**: Temporal probabilistic model with hidden states and observed emissions.

### 11.4.1 HMM Structure

**Components**:
- **Hidden states**: $S_1, S_2, ..., S_T$ (not directly observed)
- **Observations**: $O_1, O_2, ..., O_T$ (observed emissions)
- **Initial probabilities**: $\pi_i = P(S_1 = i)$
- **Transition probabilities**: $a_{ij} = P(S_t = j | S_{t-1} = i)$
- **Emission probabilities**: $b_i(o) = P(O_t = o | S_t = i)$

**Assumptions**:
1. **Markov property**: $P(S_t | S_1, ..., S_{t-1}) = P(S_t | S_{t-1})$
2. **Output independence**: $P(O_t | S_1, ..., S_T, O_1, ..., O_{t-1}) = P(O_t | S_t)$

**Joint probability**:
$$P(S_{1:T}, O_{1:T}) = P(S_1) P(O_1|S_1) \prod_{t=2}^{T} P(S_t|S_{t-1}) P(O_t|S_t)$$

### 11.4.2 Three Fundamental Problems

**1. Evaluation**: Compute $P(O_{1:T})$ given model
- **Solution**: Forward algorithm

**2. Decoding**: Find most likely state sequence given observations
- **Solution**: Viterbi algorithm

**3. Learning**: Learn parameters from data
- **Solution**: Baum-Welch (EM) algorithm

### 11.4.3 Forward Algorithm

**Goal**: Compute $P(O_{1:T})$ efficiently

**Forward variable**: $\alpha_t(i) = P(O_{1:t}, S_t = i)$

```
Algorithm: FORWARD(observations O, model λ)

begin
    N ← number of states
    T ← length of observations
    
    // Initialization
    for i = 1 to N do
α_1(i) ← π_i × b_i(O_1)
    
    // Recursion
    for t = 2 to T do
        for j = 1 to N do
α_t(j) ← b_j(O_t) × ∑_{i=1}^{N} α_{t-1}(i) × a_{ij}
    
    // Termination
    P(O) ← ∑_{i=1}^{N} α_T(i)
    
    return P(O), α
end
```

**Complexity**: $O(N^2 T)$ vs $O(N^T)$ for naive approach

### 11.4.4 Backward Algorithm

**Backward variable**: $\beta_t(i) = P(O_{t+1:T} | S_t = i)$

```
Algorithm: BACKWARD(observations O, model λ)

begin
    N ← number of states
    T ← length of observations
    
    // Initialization
    for i = 1 to N do
β_T(i) ← 1
    
    // Recursion (backwards)
    for t = T-1 down to 1 do
        for i = 1 to N do
β_t(i) ← ∑_{j=1}^{N} a_{ij} × b_j(O_{t+1}) × β_{t+1}(j)
    
    return β
end
```

**Use**: Combined with forward for parameter learning

### 11.4.5 Viterbi Algorithm

**Goal**: Find most likely state sequence

$$S^* = \arg\max_S P(S_{1:T} | O_{1:T})$$

**Viterbi variable**: $\delta_t(i) = \max_{S_{1:t-1}} P(S_{1:t-1}, S_t=i, O_{1:t})$

```
Algorithm: VITERBI(observations O, model λ)

begin
    N ← number of states
    T ← length of observations
    
    // Initialization
    for i = 1 to N do
δ_1(i) ← π_i × b_i(O_1)
ψ_1(i) ← 0
    
    // Recursion
    for t = 2 to T do
        for j = 1 to N do
δ_t(j) ← b_j(O_t) × max_i [δ_{t-1}(i) × a_{ij}]
ψ_t(j) ← argmax_i [δ_{t-1}(i) × a_{ij}]  // Backpointer
    
    // Termination
    P* ← max_i δ_T(i)
    S_T* ← argmax_i δ_T(i)
    
    // Backtracking
    for t = T-1 down to 1 do
        S_t* ← ψ_{t+1}(S_{t+1}*)
    
    return S_{1:T}*, P*
end
```

**Complexity**: $O(N^2 T)$

**Applications**:
- Speech recognition (phoneme sequence)
- Part-of-speech tagging
- Gene finding
- Activity recognition

## 11.5 EM Algorithm for HMMs

**Baum-Welch Algorithm**: EM for learning HMM parameters

### 11.5.1 Problem Setup

**Given**: Observation sequences $O^{(1)}, ..., O^{(K)}$

**Goal**: Learn $\lambda = (\pi, A, B)$ to maximize $P(O | \lambda)$

**Challenge**: Hidden states are latent variables

### 11.5.2 EM Framework

**E-step**: Compute expected sufficient statistics

**Transition counts**:
$$\xi_t(i,j) = P(S_t=i, S_{t+1}=j | O, \lambda)$$

$$\xi_t(i,j) = \frac{\alpha_t(i) a_{ij} b_j(O_{t+1}) \beta_{t+1}(j)}{P(O)}$$

**State occupation**:
$$\gamma_t(i) = P(S_t=i | O, \lambda) = \sum_j \xi_t(i,j)$$

**M-step**: Re-estimate parameters

**Initial probabilities**:
$$\hat{\pi}_i = \gamma_1(i)$$

**Transition probabilities**:
$$\hat{a}_{ij} = \frac{\sum_{t=1}^{T-1} \xi_t(i,j)}{\sum_{t=1}^{T-1} \gamma_t(i)}$$

**Emission probabilities**:
$$\hat{b}_i(v_k) = \frac{\sum_{t:O_t=v_k} \gamma_t(i)}{\sum_{t=1}^{T} \gamma_t(i)}$$

### 11.5.3 Baum-Welch Algorithm

```
Algorithm: BAUM-WELCH(observations O, num_states N, max_iter)

begin
    // Initialize parameters randomly
λ ← random_initialization(N)
    
    for iteration = 1 to max_iter do
        // E-step: Forward-backward
α ← FORWARD(O, λ)
β ← BACKWARD(O, λ)
        
        // Compute ξ and γ
        for t = 1 to T-1 do
            for i = 1 to N do
                for j = 1 to N do
ξ_t(i,j) ← α_t(i) a_{ij} b_j(O_{t+1}) β_{t+1}(j) / P(O)
        
        for t = 1 to T do
            for i = 1 to N do
γ_t(i) ← ∑_j ξ_t(i,j)
        
        // M-step: Re-estimate
        for i = 1 to N do
π_i ← γ_1(i)
        
        for i = 1 to N do
            for j = 1 to N do
                a_{ij} ← ∑_t ξ_t(i,j) / ∑_t γ_t(i)
        
        for i = 1 to N do
            for each symbol v_k do
                b_i(v_k) ← ∑_{t: O_t=v_k} γ_t(i) / ∑_t γ_t(i)
        
        // Check convergence
        if log-likelihood change < threshold then
            break
    
    return λ
end
```

**Convergence**: Guaranteed to increase likelihood (local maximum)

## 11.6 Markov Random Fields

**Markov Random Field (MRF)**: Undirected graphical model

### 11.6.1 Structure

**Undirected graph**:
- Nodes: Random variables
- Edges: Direct dependencies (symmetric)

**Cliques**: Fully connected subgraphs
- Maximal clique: Cannot be extended

### 11.6.2 Factorization

**Gibbs distribution**:
$$P(X_1, ..., X_n) = \frac{1}{Z} \prod_{C \in \mathcal{C}} \psi_C(X_C)$$

where:
- $\mathcal{C}$: Set of maximal cliques
- $\psi_C$: Potential function (non-negative)
- $Z$: Partition function (normalization constant)

$$Z = \sum_{x_1, ..., x_n} \prod_{C \in \mathcal{C}} \psi_C(x_C)$$

**Energy formulation**:
$$P(X) = \frac{1}{Z} e^{-E(X)}$$

where $E(X) = -\sum_C \log \psi_C(X_C)$

### 11.6.3 Markov Properties

**Pairwise Markov Property**:
$$X_i \perp X_j | X_{V \setminus \{i,j\}} \quad \text{if no edge between } i \text{ and } j$$

**Local Markov Property**:
$$X_i \perp X_{V \setminus (N(i) \cup \{i\})} | X_{N(i)}$$

where $N(i)$ is neighbors of $i$

**Global Markov Property**:
$$X_A \perp X_B | X_C \quad \text{if } C \text{ separates } A \text{ and } B$$

### 11.6.4 Applications

**Image segmentation**:
- Pixels as nodes
- Potentials encourage smoothness

**Protein structure prediction**:
- Amino acids as nodes
- Potentials from energy functions

**Spatial data modeling**:
- Locations as nodes
- Potentials capture spatial correlation

## 11.7 Summary

### Key Concepts

**Bayesian Networks**:
- Directed acyclic graphs
- Conditional probability distributions
- Factorization: $P(X) = \prod_i P(X_i | Pa(X_i))$
- Efficient inference with structure

**Inference Algorithms**:
- Enumeration: Brute force, exponential
- Variable Elimination: Exploit independence
- Complexity depends on tree-width

**Hidden Markov Models**:
- Temporal probabilistic models
- Hidden states, observed emissions
- Three problems: Evaluation, Decoding, Learning

**HMM Algorithms**:
- Forward: $O(N^2 T)$ for likelihood
- Viterbi: $O(N^2 T)$ for best sequence
- Baum-Welch: EM for parameter learning

**Markov Random Fields**:
- Undirected graphs
- Potential functions on cliques
- Partition function for normalization

### Algorithm Comparison

| Algorithm | Problem | Complexity | Output |
|-----------|---------|------------|--------|
| Enumeration | Any query | $O(2^n)$ | Exact |
| Variable Elim | Any query | $O(2^w n)$ | Exact |
| Forward | P(observations) | $O(N^2 T)$ | Exact |
| Viterbi | Best sequence | $O(N^2 T)$ | Exact |
| Baum-Welch | Learn params | $O(N^2 T I)$ | Local opt |

where $n$ = variables, $w$ = tree-width, $N$ = states, $T$ = time steps, $I$ = iterations

### Model Selection Guide

**Use Bayesian Networks when**:
- Clear causal structure
- Directed dependencies
- Hierarchical relationships

**Use HMMs when**:
- Sequential/temporal data
- Hidden states evolve over time
- Observations depend on current state

**Use MRFs when**:
- Symmetric relationships
- Spatial dependencies
- No natural causal direction

### Practical Considerations

1. **Structure learning**: Finding optimal graph is hard
2. **Parameter learning**: EM for hidden variables
3. **Inference**: Exact is often intractable
4. **Approximate inference**: Sampling, variational methods
5. **Scalability**: Specialized algorithms for large networks

## 11.8 Implementation

For complete Python implementations, see:

[ch11_probabilistic_graphical_models_implementation.ipynb](ch11_probabilistic_graphical_models_implementation.ipynb)

The implementation notebook includes:

**Bayesian Networks**:
1. Network construction
2. CPD specification
3. Enumeration inference
4. Variable elimination
5. Medical diagnosis example

**Hidden Markov Models**:
1. Forward algorithm
2. Backward algorithm
3. Viterbi algorithm
4. Baum-Welch training
5. Part-of-speech tagging
6. Speech recognition

**Markov Random Fields**:
1. MRF construction
2. Potential functions
3. Inference with message passing
4. Image denoising

**Applications**:
1. Medical diagnosis system
2. Weather prediction
3. Named entity recognition
4. Robot localization

## Further Reading

### Textbooks

- **Koller, D., & Friedman, N. (2009).** *Probabilistic Graphical Models*. MIT Press. [THE definitive reference]
- Aggarwal, C. C. (2021). *Artificial Intelligence: A Textbook*. Springer. [Chapter 11]
- Murphy, K. P. (2012). *Machine Learning: A Probabilistic Perspective*. MIT Press.
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer. [Chapters 8, 13]

### Classic Papers

**Bayesian Networks**:
- Pearl, J. (1988). *Probabilistic Reasoning in Intelligent Systems*. Morgan Kaufmann.
- Lauritzen, S. L., & Spiegelhalter, D. J. (1988). Local computations with probabilities. *Journal of the Royal Statistical Society*.

**Hidden Markov Models**:
- Rabiner, L. R. (1989). A tutorial on hidden Markov models. *Proceedings of the IEEE*.
- Baum, L. E., et al. (1970). A maximization technique occurring in the statistical analysis of probabilistic functions. *The Annals of Mathematical Statistics*.

**Inference**:
- Pearl, J. (1986). Fusion, propagation, and structuring in belief networks. *Artificial Intelligence*.
- Zhang, N. L., & Poole, D. (1996). Exploiting causal independence in Bayesian network inference. *JAIR*.